In [25]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import importlib


## config

In [ ]:
# ===== USER SETTINGS =====
FOLDER_A = Path("outputs/2_fairness/LNS/1000_gini")
FOLDER_B = Path("outputs/2_fairness/LNS/1000_minmax")
FOLDER_BASE = Path("outputs/2_fairness/LNS/1000_jain")

LABEL_A = "Gini"
LABEL_B = "MinMax"
LABEL_BASE = "Jain"

OUTPUT_DIR = Path("outputs/2_fairness/LNS/plots")
OUTPUT_DIR.mkdir(exist_ok=True)

In [27]:
import eval_fairness
import importlib
import pandas as pd

importlib.reload(eval_fairness)
evaluate_folder = eval_fairness.evaluate_folder

df_a = pd.DataFrame(eval_fairness.evaluate_folder(FOLDER_A))
df_b = pd.DataFrame(eval_fairness.evaluate_folder(FOLDER_B))
df_base = pd.DataFrame(eval_fairness.evaluate_folder(FOLDER_BASE))

df_a["method"] = LABEL_A
df_b["method"] = LABEL_B
df_base["method"] = LABEL_BASE

common = (
    set(df_a["instance"])
    & set(df_b["instance"])
    & set(df_base["instance"])
)

df = pd.concat([
    df_a[df_a["instance"].isin(common)],
    df_b[df_b["instance"].isin(common)],
    df_base[df_base["instance"].isin(common)],
], ignore_index=True)

print(f"Common instances: {len(common)}")


Common instances: 29


In [28]:
plt.figure(figsize=(6, 4))

df.boxplot(
    column="total_cost",
    by="method",
    grid=False,
)

plt.xlabel("Fairness objective")
plt.ylabel("Total route cost")
plt.title("Cost distribution by fairness objective")
plt.suptitle("")
plt.grid(axis="y", linestyle="--", alpha=0.5)

plt.savefig(OUTPUT_DIR / "cost_by_fairness_boxplot.png",
            dpi=300, bbox_inches="tight")
plt.close()


<Figure size 600x400 with 0 Axes>

In [29]:
plt.figure(figsize=(6, 4))

for method, dfi in df.groupby("method"):
    plt.scatter(
        dfi["total_cost"],
        dfi["fairness"],
        label=method,
        alpha=0.6,
        s=30
    )

plt.xlabel("Total cost")
plt.ylabel("Fairness")
plt.title("Cost-fairness trade-off")
plt.legend()
plt.grid(True)

plt.savefig(OUTPUT_DIR / "cost_fairness_tradeoff_all.png",
            dpi=300, bbox_inches="tight")
plt.close()


In [30]:
import numpy as np
import matplotlib.pyplot as plt

baseline = "Jain"

pivot_cost = df.pivot(
    index="instance",
    columns="method",
    values="total_cost"
)

weights = df.groupby("instance")["fairness_weight"].first()

plt.figure(figsize=(6, 4))

for method in pivot_cost.columns:
    if method == baseline:
        continue

    ratio = pivot_cost[method] / pivot_cost[baseline]

    # scatter
    plt.scatter(
        weights,
        ratio,
        alpha=0.6,
        s=30,
        label=method
    )

    # trendline (least squares)
    coef = np.polyfit(weights, ratio, 1)
    x = np.linspace(weights.min(), weights.max(), 100)
    plt.plot(
        x,
        coef[0] * x + coef[1],
        linestyle="--"
    )

plt.xlabel("Fairness weight")
plt.ylabel(f"Cost ratio (relative to {baseline})")
plt.title("Cost penalty vs fairness preference")
plt.legend()
plt.grid(True)

plt.savefig(
    OUTPUT_DIR / "cost_ratio_vs_weight_trend.png",
    dpi=300,
    bbox_inches="tight",
)
plt.close()


In [31]:
from scipy.stats import friedmanchisquare

pivot_cost = df.pivot(
    index="instance",
    columns="method",
    values="total_cost"
)

stat, p = friedmanchisquare(
    *[pivot_cost[c] for c in pivot_cost.columns]
)

print(f"Friedman test (cost): statistic={stat:.3f}, p-value={p:.4e}")


Friedman test (cost): statistic=48.483, p-value=2.9655e-11


In [32]:
from scipy.stats import wilcoxon
from itertools import combinations

alpha = 0.05
methods = pivot_cost.columns
m = len(methods)
bonferroni = alpha / (m * (m - 1) / 2)

print(f"Bonferroni-corrected alpha = {bonferroni:.4f}\n")

for a, b in combinations(methods, 2):
    stat, p = wilcoxon(pivot_cost[a], pivot_cost[b])
    sig = "YES" if p < bonferroni else "NO"
    print(f"{a} vs {b}: p={p:.4e}  significant={sig}")


Bonferroni-corrected alpha = 0.0167

Gini vs Jain: p=3.7253e-09  significant=YES
Gini vs MinMax: p=9.8225e-05  significant=YES
Jain vs MinMax: p=3.7253e-09  significant=YES


In [33]:
from scipy.stats import spearmanr

baseline = "Jain"
pivot_cost = df.pivot(index="instance", columns="method", values="total_cost")
weights = df.groupby("instance")["fairness_weight"].first()

for method in pivot_cost.columns:
    if method == baseline:
        continue

    ratio = pivot_cost[method] / pivot_cost[baseline]
    rho, p = spearmanr(weights, ratio)

    print(
        f"{method} vs {baseline}: "
        f"Spearman rho={rho:.3f}, p={p:.4e}"
    )


Gini vs Jain: Spearman rho=0.096, p=6.2007e-01
MinMax vs Jain: Spearman rho=0.348, p=6.4672e-02


In [34]:
pivot_time = df.pivot(
    index="instance",
    columns="method",
    values="execution_time"
)

plt.figure(figsize=(6, 4))

for method in pivot_time.columns:
    times = pivot_time[method]

    plt.scatter(
        weights,
        times,
        alpha=0.6,
        s=30,
        label=method
    )

    # trendline
    coef = np.polyfit(weights, times, 1)
    x = np.linspace(weights.min(), weights.max(), 100)
    plt.plot(
        x,
        coef[0] * x + coef[1],
        linestyle="--"
    )

plt.xlabel("Fairness weight")
plt.ylabel("Execution time [ms]")
plt.title("Runtime vs fairness preference")
plt.legend()
plt.grid(True)

plt.savefig(
    OUTPUT_DIR / "runtime_vs_weight_trend.png",
    dpi=300,
    bbox_inches="tight",
)
plt.close()
